## Neural Network from scratch for L layer

Problem:
We have given M training set already X = [X1, X2, X3, .... XM]
and actual label set Y = [Y1, Y2, .... YM]

want to build a neural network which will have hidden layers as relu and output layer as sigmoid. Basically a classifier neural network model.

##Interfaces for understanding the blueprint

In [1]:
import numpy as np
from abc import ABC, abstractmethod

class Layer(ABC):

    @abstractmethod
    def apply_activation(self, Z: np.ndarray) -> np.ndarray: ...

    @abstractmethod
    def forward(self, A_prev: np.ndarray) -> np.ndarray:
        """
        Forward pass for this layer.

        Args:
            A_prev: activations from previous layer, shape (N_{L-1}, m)

        Returns:
            A: activations of this layer, shape (N_L, m)
        """
        ...

    #Perform the backword on Layer L which will get dA_L from L + 1 layer
    # Z_1 -> A_1 -> Z_2 -> A_2 -> ... Z_L -> A_L -> Z_L+1 -> A_L+1 -> ........
    # so our target is to get dA_L, dB_L1, dW_L for a layer
    # dA_L = dL/dZ_L+1 * dZ_L+1/dA_L
    # So each layer will calulate dA_L-1, dW_L, dB_L
    @abstractmethod
    def backward(self, dA_L: np.ndarray):
        """
        Backward pass for this layer.

        Args:
            dA_L: gradient of loss w.r.t. this layer's output, shape (N_L, m)

        Returns:
            (dA_prev, dW, dB):
                dA_prev: grad w.r.t. previous layer's activation, shape (N_{L-1}, m)
                dW:      grad w.r.t. weights, shape (N_L, N_{L-1})
                dB:      grad w.r.t. bias,    shape (N_L, 1)
        """
        ...

In [2]:
class Dense(Layer):
    def __init__(self, current_layer_units, previous_layer_units, activation):
        self.W = np.random.randn(current_layer_units, previous_layer_units) * np.sqrt(2 / previous_layer_units)
        self.B = np.zeros((current_layer_units,1))
        self.Z = None
        self.A = None
        self.A_prev = None
        self.dW = None
        self.dB = None
        self.activation = activation

    def linear_forward(self, A_prev: np.ndarray):
        return np.dot(self.W, A_prev) + self.B

    def apply_activation(self, Z):
        if self.activation == "relu":
            return np.maximum(0, Z)
        elif self.activation == "sigmoid":
            return 1 / (1 + np.exp(-Z))
        else:
            raise ValueError(f"unknown activation: {self.activation}")

    def forward(self, A_prev):
        self.A_prev = A_prev
        self.Z = self.linear_forward(A_prev=A_prev)
        self.A = self.apply_activation(self.Z)
        return self.A

    def backward(self, dA_L):
        m = self.A_prev.shape[1]

        if self.activation == "sigmoid":
            dZ = dA_L * (self.A * (1 - self.A))
        else:
            dZ = dA_L * (self.Z > 0)
        # (units, m) @ (prev_units, m) → inner dims m and prev_units don't match → ERROR, so we transpose and it dW = (units, prev_units)
        self.dW = dZ @ self.A_prev.T / m # dW = dL/dZ * dZ/da_prev
        self.dB = np.sum(dZ, axis=1, keepdims=True) / m
        return self.W.T @ dZ

class NeuralNetwork:
    def __init__(self, input_features, y, layer_dims, mini_batch):
        self.layers = []
        self.input_features = input_features
        self.y = y
        self.mini_batch = mini_batch
        self.layer_dims = layer_dims
        for i in range(1, len(layer_dims)):
            activation = "sigmoid" if i == len(layer_dims)-1 else "relu"
            self.add_layer(Dense(layer_dims[i], layer_dims[i-1], activation))

    def add_layer(self, layer):
        self.layers.append(layer)

    def forward(self, X):
        A = X
        for layer in self.layers:
            A = layer.forward(A)
        return A

    def backward(self, dA_L):
        for layer in reversed(self.layers):
            dA_L = layer.backward(dA_L)

    def update(self, lr):
        for layer in self.layers:
            layer.W -= layer.dW * lr
            layer.B -= layer.dB * lr

    def train(self, epochs, compute_loss, loss_gradient, lr):
        m = self.input_features.shape[1]
        for epoch in range(epochs):
            perm = np.random.permutation(m)
            X_sh, y_sh = self.input_features[:, perm], self.y[:, perm]
            for start in range(0, m, self.mini_batch):
                end = start + self.mini_batch
                X_b = X_sh[:, start:end]
                y_b = y_sh[:, start:end]

                A = self.forward(X_b)
                loss = compute_loss(A, y_b)
                dA = loss_gradient(A, y_b)
                self.backward(dA)
                self.update(lr)

            if epoch % 100 == 0:
                A_full = self.forward(self.input_features)
                full_loss = compute_loss(A_full, self.y)
                print(f"epoch {epoch}: loss {full_loss:.4f}")

In [3]:
# ---------- Binary classification: sigmoid + Binary Cross-Entropy ----------
def bce_loss(A, y):
    m = y.shape[1]
    eps = 1e-8                                   # clip to avoid log(0)
    A = np.clip(A, eps, 1 - eps)
    return -np.sum(y*np.log(A) + (1-y)*np.log(1-A)) / m

def bce_grad(A, y):                              # dL/dA
    m = y.shape[1]
    eps = 1e-8
    A = np.clip(A, eps, 1 - eps)
    return -(y/A - (1-y)/(1-A))

# ---------- Regression: linear + Mean Squared Error ----------
def mse_loss(A, y):
    m = y.shape[1]
    return np.sum((A - y)**2) / (2*m)            # the /2 makes the grad clean

def mse_grad(A, y):                              # dL/dA
    m = y.shape[1]
    return (A - y) / m

In [4]:
# ---------- synthetic data: two Gaussian blobs (binary classification) ----------
# NOTE: features are ROWS, examples are COLUMNS -> shape (features, m),
# matching the m = shape[1] convention used everywhere in the layer code.
np.random.seed(42)

m = 400                       # total examples
half = m // 2

# class 0 centered at (-2,-2), class 1 centered at (+2,+2)
X0 = np.random.randn(2, half) + np.array([[-2], [-2]])           # (2, 200)
X1 = np.random.randn(2, half) + np.array([[ 2], [ 2]])          # (2, 200)

X = np.hstack([X0, X1])                                          # (2, 400)
y = np.hstack([np.zeros((1, half)), np.ones((1, half))])        # (1, 400)

# shuffle examples (columns) so classes are interleaved
perm = np.random.permutation(m)
X = X[:, perm]
y = y[:, perm]

print("X", X.shape, "y", y.shape)          # X (2, 400) y (1, 400)

X (2, 400) y (1, 400)


In [5]:
# ---------- build & train ----------
nn = NeuralNetwork(X, y, [2, 5, 1], 10)
    # output: 1 unit,  fan-in 8

nn.train(epochs=1000, compute_loss=bce_loss, loss_gradient=bce_grad, lr=0.05)

# ---------- train accuracy ----------
preds = (nn.forward(X) > 0.5).astype(int)
acc = np.mean(preds == y)
print(f"\ntrain accuracy: {acc*100:.1f}%")

epoch 0: loss 0.0964


epoch 100: loss 0.0074


epoch 200: loss 0.0051


epoch 300: loss 0.0042


epoch 400: loss 0.0037


...

epoch 600: loss 0.0030


epoch 700: loss 0.0027


epoch 800: loss 0.0023


epoch 900: loss 0.0020



train accuracy: 100.0%
